# Lab: Cortex Search Part 2

**This is an OPTIONAL lab. We will NOT dedicate class time for this lab. We recommend you try this lab outside of the dedicated class time. If you happen to have free extra time leftover during the class, please try this out as you wish!**

📚  In this lab you will learn and practice the following:

Part 2: Cortex Search Engine for structured and semi-structured data

   ❄️ Create a Cortex Search Service over structured/semi-structured data

   ❄️ Query the Cortex Search Service using SEARCH_PREVIEW

   ❄️ Use filter expressions to narrow search results

   ❄️ Test the search service interactively in the Cortex Search Playground

📌 **Note**:

Due to **regional workload spikes**, there may be **latency** with some of the steps. In the real world, for consistent performance, customers can explore [**Provisioned Throughput**](https://docs.snowflake.com/en/user-guide/snowflake-cortex/provisioned-throughput).

If you find your queries are running for more than 5 minutes, cancel and come back and try them later.

If you find models that are deprecated, use CoCo to help you fix the issue by selecting a suitable model.

---

### 🤖 Use CoCo as you go!

> **💡 TIP 1**: Use CoCo to explain complex SQL statements. Select any query and ask *"Explain this SQL"* to get a plain-language breakdown of what it does.
>
> **💡 TIP 2**: Want to learn more about any function? Ask CoCo *"What does [function name] do?"* to get its syntax, supported options, and examples.
>
> **💡 TIP 3**: If you encounter a deprecated model error, ask CoCo *"Replace deprecated models in this notebook with current similar low-cost alternatives"* and it will fix them for you.

---

## Introduction

Free text columns in structured tables often contain valuable information that needs to be searched and analyzed. We can use Cortex Search to index free text in structured tables, then combine it with CORTEX.COMPLETE to answer natural language questions, all directly in SQL.

In this lab, we will:
1. Build a search service on traveler activity reviews
2. Use SEARCH_PREVIEW to verify retrieval and answer questions with RAG
3. Use CORTEX_SEARCH_BATCH to match traveler personas to the best activities at scale

Note: For more advanced structured table analysis across all columns, you will use **Cortex Analyst**, which will be covered in the next module.

## Connect to a Service

Before running cells in this notebook, you must connect to a compute service.

**First time (create a new service):**
1. Click the **Connect** button at the top of this notebook
2. Click **Create Service** — a default name like `{{user}}_SERVICE1` will be suggested
3. Click **Service Settings** and select `ALLOW_ALL_EAI` as the external access integration
4. Leave other settings as default and click **Create**
5. Wait for the service to reach a **READY** state

**Returning (service already exists):**
1. Click the **Connect** button
2. Select your existing service from the list

Once connected, you can run Python and SQL cells interactively.

### Building a Cortex Search Service on traveler activity reviews

📌 **Note:** 

Ensure your role has been granted the `SNOWFLAKE.CORTEX_USER` database role to use Cortex Search services.

### Set up your current context for the role, database, schema and warehouse.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
user = session.get_current_user().strip('"')
your_db = user + '_genai_db'
print('Your current CONTEXT information:')
print(session)

In [ ]:
%%sql -r Set_up_your_current_context_for_the_sql
USE ROLE genai_role;
USE DATABASE {{user}}_genai_db;
USE SCHEMA raw;
USE WAREHOUSE {{user}}_genai_wh;
ALTER SESSION SET query_tag = '{{user}} lab - TOPIC: Cortex Search Part 2';
SHOW PARAMETERS LIKE 'query_tag' in session 
  ->> SELECT "value" AS query_tag FROM $1;

### Searching traveler_activity table.

The traveler_activity table contains review_text and a few other key fields that would be helpful for Travelbug. We can build a search service on top of it to find answers.

In [ ]:
%%sql -r Searching_traveler_activity_table_sql
SELECT  activity_name, 
        activity_location,
        review_text
FROM {{user}}_genai_db.presentation.traveler_activity
LIMIT 5;


### Enable change tracking for the traveler_activity table.

We need to enable change tracking for the Cortex Search Service so that any updates, inserts, or deletions made to the structured table are automatically reflected in the search index. This ensures that the search results remain accurate and up to date without requiring a full reindexing, improving performance and efficiency.

In [ ]:
%%sql -r Enable_change_tracking_for_the_sql
USE ROLE genai_role;
ALTER TABLE {{user}}_genai_db.presentation.traveler_activity 
SET CHANGE_TRACKING = TRUE;  


### Create a Cortex Search Service on a structured table.

In this example, we are using the traveler_activity table and building a search service on the review_text column. It can filter on the activity_name column. Optionally, we can specify the embedding model name. 

In [ ]:
%%sql -r Create_a_Cortex_Search_Service_on_a_sql
USE ROLE genai_role;
-- Create a Cortex Search Service
CREATE OR REPLACE CORTEX SEARCH SERVICE {{user}}_genai_db.resources.TRAVELBUG_SEARCH_STRUCTURED
  ON review_text
  ATTRIBUTES activity_name
  WAREHOUSE = {{user}}_genai_wh
  TARGET_LAG = '30 minutes'
  EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
  AS (
    SELECT
       activity_id || '_' || review_id AS record_id,
       activity_name,
       activity_location,
       review_text
    FROM  {{user}}_genai_db.presentation.traveler_activity
);

## Extracting vector embeddings

### Using the cortex_search_data_scan function to see detailed vector embeddings.

We can verify the data indexed by a Cortex Search Service, including the columns defined in the source query and the computed vector embeddings for the search column.

The function returns all columns specified in the source query, along with the embeddings for the search column. The embedding column is of the VECTOR data type and is named _GENERATED_EMBEDDINGS_{MODEL_NAME}. 

Vector embeddings are numerical representations of data, often used in machine learning. Each vector is a point in a high-dimensional space, and its values correspond to specific coordinates in that space. The "distance" between vectors indicates their similarity. To measure this similarity, we use cosine similarity.

Cosine similarity calculates the angle between two vectors:

A value close to 1 means the vectors are very similar and point in the same direction.
A value close to -1 means the vectors are opposites.
A value close to 0 means the vectors are orthogonal (not related).
In essence, vector embeddings turn complex data into numerical forms that can be compared based on their relative positions in this space, helping us identify similarities and differences between them.

In [ ]:
%%sql -r Using_the_cortex_search_data_scan_sql
USE ROLE genai_role;
USE DATABASE {{user}}_genai_db;
USE SCHEMA resources;
SELECT  *
FROM
  TABLE (
    CORTEX_SEARCH_DATA_SCAN (
      SERVICE_NAME => 'TRAVELBUG_SEARCH_STRUCTURED'
    ));

### Review the token counts for review_text.
We can use the **CORTEX.COUNT_TOKENS** command to review the tokens for the review_text. As you can see the numbers are in the low end of the scale.

📌 **Note:** 

For best search results with Cortex Search, Snowflake recommends splitting text into chunks of no more than 512 tokens (about 385 English words, or ~2048 characters). For longer text, chunking helps ensure better search quality by making sure all content is properly indexed.

In [ ]:
%%sql -r Review_the_token_counts_for_review_text_sql
SELECT review_text, 
       SNOWFLAKE.CORTEX.COUNT_TOKENS('snowflake-arctic-embed-m', review_text) as token_count
FROM {{user}}_genai_db.presentation.traveler_activity
ORDER BY token_count DESC
LIMIT 5;

### Verifying search with SEARCH_PREVIEW

Use the `SEARCH_PREVIEW` function to quickly verify that the search service is returning relevant results. This gives you the raw search output: the retrieved documents ranked by relevance.

In [ ]:
%%sql -r Using_search_preview_to_confirm_the_sql
SELECT PARSE_JSON(
  SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
      '{{user}}_genai_db.resources.TRAVELBUG_SEARCH_STRUCTURED',
      '{
        "query": "What are the top two reviews for Mountain Hiking",
        "columns":[
            "record_id",
            "review_text",
            "activity_name"
        ],
        "filter": {"@eq": {"activity_name": "Mountain Hiking"} },
        "limit":1
      }'
  )
)['results'] as results;

In [ ]:
# Format the previous cell output
import json
from IPython.display import display, HTML

result = session.sql(f"""
SELECT PARSE_JSON(
  SNOWFLAKE.CORTEX.SEARCH_PREVIEW(

'{user}_genai_db.resources.TRAVELBUG_SEARCH_STRUCTURED',
      '{{
        "query": "What are the top two reviews for Mountain Hiking",
        "columns":["record_id", "review_text", "activity_name"],
        "filter": {{"@eq": {{"activity_name": "Mountain Hiking"}} }},
        "limit":2
      }}'
  )
)['results'] as results
""").collect()

parsed = json.loads(result[0]['RESULTS'])

html = """<style>
.sp { margin: 20px 0; padding: 16px; border: 1px solid #d0d0d0; border-radius: 10px; background: #fafafa; font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, "Helvetica Neue", Arial, sans-serif; font-size: 14px; }
.sp h4 { font-family: inherit; margin-top: 0; }
.sp .card { margin: 10px 0; padding: 12px; border: 1px solid #e0e0e0; border-radius: 8px; background: #fff; }
</style>"""
html += '<div class="sp"><h4>\U0001f50d Search Preview Results</h4>'
for i, r in enumerate(parsed):
    html += '<div class="card">'
    html += f'<b>Result {i+1}</b> (Record: {r.get("record_id", "N/A")})<br>'
    html += f'<b>Activity:</b> {r.get("activity_name", "N/A")}<br>'
    html += f'<b>Review:</b> {r.get("review_text", "N/A")}<br>'
    if "@scores" in r:
        scores = r["@scores"]
        html += f'<b>Scores:</b> cosine={scores.get("cosine_similarity", "N/A"):.4f}, reranker={scores.get("reranker_score", "N/A"):.4f}'
    html += '</div>'
html += '</div>'

display(HTML(html))

### Test your Search Service in the Cortex Search Playground

Now that the search service is created, you can also test it interactively using the built-in **Search Playground** in Snowsight — no code required.

### Steps:

1. In Snowsight, navigate to **AI & ML** > **Cortex AI** > **Search** from the left sidebar
2. You will see your **TRAVELBUG_SEARCH_STRUCTURED** service listed
3. Click on the service name to open the **Search Playground**
4. Type a question in the search box and press Enter

### Try these questions:

- **What are the best mountain hiking experiences?**
- **Were there any complaints about food during activities?**
- **Which water activities got the best reviews?**
- **What did travelers say about the sunset boat cruise?**

---

📌 **Tip:** The Playground provides the same retrieval as `SEARCH_PREVIEW` but through a point-and-click interface. It's useful for quick interactive testing and demos without writing SQL. No LLM is involved — it only retrieves and ranks relevant chunks.

## How to decide whether to chunk data

One common question that arises is whether we should chunk the text for Cortex Search or if it is unnecessary when using a large LLM model such as Claude 4 Sonnet.

If you do not chunk the text, the embedding model will only process the first 512 tokens. When you run RAG with a large language model, it will have access to the full context, but the model may get distracted, potentially leading to worse retrieval performance at the full document level.

More details are in the Snowflake documentation: 

https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/cortex-search-overview

### Splitting text to improve the quality of the search in a structured table.
Suppose we have larger review text in our structured table. How do we split it?  In our case the the reviews tables is far too small and does not need chunking generally.

Below is an example of creating a new table with the structured text and loading data from the traveler_activity table by chunking the text first using **SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER** function.

In [ ]:
%%sql -r Splitting_text_to_improve_the_quality_sql
-- Create table with chunked data
CREATE OR REPLACE TABLE {{user}}_genai_db.raw.traveler_activity_structured_chunked (
    chunk_id,
    activity_name,
    activity_location,
    chunk
) AS
SELECT
   activity_id || '_' || review_id || '_' || c.index AS chunk_id,
   activity_name,
   activity_location,
   c.value::VARCHAR AS chunk
FROM
  {{user}}_genai_db.presentation.traveler_activity,
  LATERAL FLATTEN(
      input => SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER(
         COALESCE(activity_name, '') || '-' || 
         COALESCE(activity_location, '') || '-' || 
         COALESCE(review_text, '') || '-' || 
         COALESCE(activity_price, ''),
         'none',
         2048,
         300
      )
  ) c;

### Creating the chunked data table: traveler_activity_structured_chunked

The following query does the following:

Extracts text from the traveler_activity table.

Creates a unique **chunk_id** by concatenating activity_id, review_id, and the chunk index.

Concatenates activity_name, activity_location, review_text, and activity_price into a single string (handling NULLs).

Splits text into **2048-character** chunks (~512 tokens) using **SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER**, following Snowflake's recommendation of no more than 512 tokens per chunk.

Ensures context retention by overlapping chunks by **300** characters.

Uses **LATERAL FLATTEN** to convert the output into separate rows.

Creates the traveler_activity_structured_chunked table in the raw schema with the processed chunks.

In [ ]:
%%sql -r Inserting_chunked_data_into_traveler_sql
-- This INSERT is now handled by the CREATE TABLE IF NOT EXISTS above
-- Keeping this cell for reference - it will be skipped if table already exists
SELECT 'traveler_activity_structured_chunked table ready' AS status;

In [ ]:
%%sql -r View_chunked_data_sql
SELECT * 
FROM {{user}}_genai_db.raw.traveler_activity_structured_chunked
LIMIT 10;

### Enabling change tracking on the chunked table.

We need to enable change tracking for the Cortex Search Service so that any updates, inserts, or deletions made to the structured table with chunked data are automatically reflected in the search index. This ensures that the search results remain accurate and up to date without requiring a full reindexing, improving performance and efficiency.

In [ ]:
%%sql -r Enabling_change_tracking_on_the_chunked_sql
ALTER TABLE {{user}}_genai_db.raw.traveler_activity_structured_chunked 
SET CHANGE_TRACKING = TRUE; 

### Creating a search service on structured table where free text column has been chunked.
If you chunked the data in a free text column similar to above, it is possible to build a search service on top of it.

In [ ]:
%%sql -r Creating_a_search_service_on_structured_sql
-- Create a Cortex Search Service
CREATE OR REPLACE CORTEX SEARCH SERVICE {{user}}_genai_db.resources.travelbug_search_structured_chunks
    ON CHUNK
    WAREHOUSE = {{user}}_genai_wh
    TARGET_LAG = '30 minutes'
    EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
    AS (
        SELECT chunk_id, activity_name, activity_location, chunk
        FROM {{user}}_genai_db.raw.traveler_activity_structured_chunked
    );

## Batch Cortex Search

`CORTEX_SEARCH_BATCH` is a table function that lets you submit a batch of queries to a Cortex Search Service. It is designed for **offline, high-throughput** use cases where you need to run many searches at once, significantly faster than calling the interactive API in a loop.

**Common use cases:**
- **Activity matching**: match trip planning requests against activity reviews to recommend experiences
- **Audience segmentation**: find which travelers share similar interests based on review text
- **Bulk enrichment**: augment traveler profiles with related activity insights from reviews
- **Clustering**: group similar reviews or activities by semantic similarity

### Create trip planning requests for batch matching.

We will create a table of trip planning requests that represent different traveler personas. Then we will use `CORTEX_SEARCH_BATCH` to match each request against the existing `TRAVELBUG_SEARCH_STRUCTURED` service (built on traveler activity reviews) to find the best matching experiences.

In [ ]:
%%sql -r create_customers_sql
-- Create a table of trip planning requests representing different traveler personas
CREATE OR REPLACE TABLE {{user}}_genai_db.raw.trip_requests (
    id INT,
    traveler_persona VARCHAR,
    request_text VARCHAR
);

INSERT INTO {{user}}_genai_db.raw.trip_requests VALUES
(1, 'Adventure Seeker', 'Looking for thrilling outdoor activities with adrenaline, heights, and physical challenges in nature'),
(2, 'Food & Wine Enthusiast', 'Want authentic local cuisine, wine tasting, and gourmet food experiences with beautiful scenery'),
(3, 'Family Traveler', 'Need safe, fun activities suitable for children and families, educational and engaging for all ages'),
(4, 'Water Sports Lover', 'Interested in ocean activities like diving, snorkeling, kayaking, and exploring marine life'),
(5, 'Relaxation Seeker', 'Looking for peaceful, calming experiences like sunset views, gentle boat rides, and scenic beauty'),
(6, 'Wildlife Explorer', 'Want to see exotic animals in their natural habitat, safari experiences, and nature photography'),
(7, 'Urban Culture Fan', 'Interested in city tours, historical landmarks, architecture, museums, and local culture'),
(8, 'Thrill Seeker', 'Want extreme sports, rock climbing, mountain hiking with difficult trails and challenging terrain');

In [ ]:
%%sql -r create_resumes_sql
-- Verify the trip requests
SELECT * FROM {{user}}_genai_db.raw.trip_requests ORDER BY id;

### Batch activity matching: match trip requests to reviews.

Use `CORTEX_SEARCH_BATCH` to match each trip planning request against the `TRAVELBUG_SEARCH_STRUCTURED` service. This finds the most relevant activity reviews for each traveler persona, demonstrating how a travel platform could power a recommendation engine.

📌 **Note:** The search service needs to be in ACTIVE state. If you get an error, wait 1-2 minutes and retry.

In [ ]:
%%sql -r create_customer_search_sql
-- Batch activity matching: find best activity reviews for each trip request
SELECT
    t.traveler_persona,
    t.request_text,
    r.activity_name AS matched_activity,
    r.activity_location AS matched_location,
    LEFT(r.review_text, 150) AS review_snippet,
    r.METADATA$RANK AS match_rank
FROM {{user}}_genai_db.raw.trip_requests AS t,
LATERAL CORTEX_SEARCH_BATCH(
    service_name => '{{user}}_genai_db.resources.TRAVELBUG_SEARCH_STRUCTURED',
    query => t.request_text
) AS r
WHERE r.METADATA$RANK <= 3  -- Top 3 matches per request
ORDER BY t.id, r.METADATA$RANK;

### Batch matching with a filter: find only water activity reviews.

We can also apply a filter during batch search to narrow results to specific activity types. Here we batch-match our water sports queries and restrict results to only water-related activities using the `@or` filter operator.

📌 **Note:** The `filter` parameter in `CORTEX_SEARCH_BATCH` must be a literal object expression; it cannot be passed dynamically from a column.

In [ ]:
%%sql -r create_resume_search_sql
-- Create focused water activity queries
CREATE OR REPLACE TEMPORARY TABLE {{user}}_genai_db.raw.water_activity_queries (
    id INT,
    scenario VARCHAR,
    query_text VARCHAR
);

INSERT INTO {{user}}_genai_db.raw.water_activity_queries VALUES
(1, 'Beginner Snorkeler', 'easy beginner-friendly underwater experience with colorful fish and coral reefs'),
(2, 'Advanced Diver', 'deep water diving with challenging conditions and rare marine life'),
(3, 'Kayak Explorer', 'peaceful kayaking through scenic waterways and coastal exploration'),
(4, 'Sunset Cruise Fan', 'romantic evening boat cruise with beautiful sunset views and relaxation');

### Run batch search filtered to water activities only.

The `@or` filter restricts results to reviews for Snorkeling Adventure, Scuba Diving, Kayaking Adventure, and Sunset Boat Cruise.

In [ ]:
%%sql -r batch_dedup_sql
-- Batch search with filter: only return water-related activity reviews
SELECT
    q.scenario,
    r.activity_name,
    r.activity_location,
    LEFT(r.review_text, 200) AS review_snippet,
    r.METADATA$RANK AS match_rank
FROM {{user}}_genai_db.raw.water_activity_queries AS q,
LATERAL CORTEX_SEARCH_BATCH(
    service_name => '{{user}}_genai_db.resources.TRAVELBUG_SEARCH_STRUCTURED',
    query => q.query_text,
    filter => {'@or': [
        {'@eq': {'activity_name': 'Snorkeling Adventure'}},
        {'@eq': {'activity_name': 'Scuba Diving'}},
        {'@eq': {'activity_name': 'Kayaking Adventure'}},
        {'@eq': {'activity_name': 'Sunset Boat Cruise'}}
    ]}
) AS r
WHERE r.METADATA$RANK <= 2  -- Top 2 per scenario
ORDER BY q.id, r.METADATA$RANK;

### Interpret the results.

Look at the batch activity matching results:
- The **Adventure Seeker** persona should match Mountain Hiking, Rock Climbing, or Safari reviews
- The **Food & Wine Enthusiast** should surface Wine Tasting reviews
- The **Water Sports Lover** should match Snorkeling, Scuba Diving, and Kayaking reviews
- The **Relaxation Seeker** should find Sunset Boat Cruise reviews
- The **Wildlife Explorer** should match Safari Adventure reviews

For the filtered water activity queries:
- Each query is scoped to a specific activity using the `filter` parameter
- Results should be the most semantically relevant reviews within that activity category

📌 **Key insight:** Batch Cortex Search processes all queries in parallel with significantly higher throughput than the interactive API. This makes it ideal for recommendation engines, personalization pipelines, and any workflow that needs to match hundreds or thousands of requests against a review corpus.

### Combining SEARCH_PREVIEW with CORTEX.COMPLETE for RAG

Now let's combine the two: use `SEARCH_PREVIEW` to retrieve relevant reviews, then pass them as context to `CORTEX.COMPLETE` so the LLM can summarize and answer the question in natural language.

This is the **Retrieval Augmented Generation (RAG)** pattern: retrieve context first, then reason over it with an LLM.

In [ ]:
%%sql -r RAG_Best_Wine_Tasting_sql
-- RAG Example 1: What are the best wine tasting reviews?
WITH search_results AS (
  SELECT PARSE_JSON(
    SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        '{{user}}_genai_db.resources.TRAVELBUG_SEARCH_STRUCTURED',
        '{
          "query": "best wine tasting experience",
          "columns": ["review_text", "activity_name", "activity_location"],
          "filter": {"@eq": {"activity_name": "Wine Tasting"} },
          "limit": 3
        }'
    )
  ) AS response
)
SELECT SNOWFLAKE.CORTEX.COMPLETE(
  'claude-haiku-4-5',
  CONCAT(
    'You are a helpful travel assistant. Based on the following reviews, answer the question.\n\n',
    'Question: What are the best wine tasting experiences based on traveler reviews?\n\n',
    'Reviews:\n', response::VARCHAR, '\n\n',
    'Provide a concise, helpful summary.'
  )
) AS answer
FROM search_results;

### RAG Example 2: Finding food-related complaints

This time we search without a filter so it spans all activities.

In [ ]:
%%sql -r RAG_Food_Complaints_sql
-- RAG Example 2: What are the most common food complaints from travelers?
WITH search_results AS (
  SELECT PARSE_JSON(
    SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        '{{user}}_genai_db.resources.TRAVELBUG_SEARCH_STRUCTURED',
        '{
          "query": "bad food experience complaint disappointing meal",
          "columns": ["review_text", "activity_name", "activity_location"],
          "limit": 5
        }'
    )
  ) AS response
)
SELECT SNOWFLAKE.CORTEX.COMPLETE(
  'claude-haiku-4-5',
  CONCAT(
    'You are a helpful travel assistant. Based on the following reviews, answer the question.\n\n',
    'Question: What are the most common food-related complaints from travelers?\n\n',
    'Reviews:\n', response::VARCHAR, '\n\n',
    'Summarize the key complaints concisely.'
  )
) AS answer
FROM search_results;

### RAG Example 3: Worst mountain hiking reviews

Combine a filter with a negative-sentiment query to find the worst reviews for a specific activity.

In [ ]:
%%sql -r RAG_Worst_Mountain_Hiking_sql
-- RAG Example 3: What are the worst mountain hiking reviews?
WITH search_results AS (
  SELECT PARSE_JSON(
    SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        '{{user}}_genai_db.resources.TRAVELBUG_SEARCH_STRUCTURED',
        '{
          "query": "terrible awful worst dangerous difficult",
          "columns": ["review_text", "activity_name", "activity_location"],
          "filter": {"@eq": {"activity_name": "Mountain Hiking"} },
          "limit": 3
        }'
    )
  ) AS response
)
SELECT SNOWFLAKE.CORTEX.COMPLETE(
  'claude-haiku-4-5',
  CONCAT(
    'You are a helpful travel assistant. Based on the following reviews, answer the question.\n\n',
    'Question: What are the worst mountain hiking experiences according to travelers?\n\n',
    'Reviews:\n', response::VARCHAR, '\n\n',
    'Provide a concise summary of the negative experiences.'
  )
) AS answer
FROM search_results;

### Experiment with your own questions

Try modifying the queries above to explore other topics. You can change:
- The `query` text to search for different topics
- The `filter` to target specific activities (e.g., `"Scuba Diving"`, `"City Tour"`, `"Safari Adventure"`)
- The `limit` to retrieve more or fewer results
- The question in the prompt to ask different things about the retrieved reviews

This pattern (**retrieve with SEARCH_PREVIEW, then reason with CORTEX.COMPLETE**) gives you full RAG capabilities directly in SQL without needing a separate application.

## SEARCH_PREVIEW vs. Playground vs. CORTEX.COMPLETE (RAG)

This lab builds on Part 1 by adding LLM generation on top of retrieval. Here's how the three approaches compare:

| | SEARCH_PREVIEW (SQL) | Playground (Snowsight UI) | SEARCH_PREVIEW + CORTEX.COMPLETE (Part 2) |
|---|---|---|---|
| **Output** | Raw ranked chunks (JSON) | Raw ranked chunks (interactive UI) | LLM-synthesized natural-language answer |
| **LLM involved?** | No — retrieval only | No — retrieval only | Yes — you chain `CORTEX.COMPLETE` to generate answers from chunks |
| **Control** | Columns, filters, limit via SQL | Point-and-click query interface | Full: choose model, write prompt, post-process |
| **Best for** | Programmatic retrieval, validation, pipelines | Quick interactive testing, demos | Production RAG — polished answers grounded in your data |

**In short:**
- `SEARCH_PREVIEW` and the **Playground** both do the same thing: retrieve and rank the most relevant chunks from your Cortex Search Service. Neither involves an LLM.
- The **RAG pattern** (Part 2) adds `CORTEX.COMPLETE` on top — it takes those retrieved chunks and passes them as context to an LLM, which generates a coherent, grounded answer.
- The examples above (wine tasting, food complaints, worst hikes) demonstrate this two-step pattern: retrieve with `SEARCH_PREVIEW`, then generate with `CORTEX.COMPLETE`.

## 🎯 Challenge Questions

Test your understanding of Cortex Search concepts covered in this lab.

In [ ]:
from snowflake.snowpark.context import get_active_session
from IPython.display import display, HTML

session = get_active_session()

quiz_data = [
    {"q": "What function allows you to match multiple queries against a Cortex Search Service in a single operation?", "options": ["A) CORTEX_SEARCH_MULTI", "B) CORTEX_SEARCH_BATCH", "C) SEARCH_PREVIEW_ALL", "D) CORTEX_BATCH_QUERY"], "hash": "02801276754bcae42b6ffae62fb79793"},
    {"q": "Which function allows you to extract and examine vector embeddings from a Cortex Search Service?", "options": ["A) SEARCH_PREVIEW", "B) VECTOR_EXTRACT", "C) CORTEX_SEARCH_DATA_SCAN", "D) EMBEDDING_LOOKUP"], "hash": "9d24631bbbc446cf3c492cc5cce49bf5"},
    {"q": "Why might you choose to chunk structured data like review text before creating a search service?", "options": ["A) To improve search result quality by creating more focused embedding segments", "B) To reduce storage costs", "C) To speed up query execution", "D) Chunking is required for all Cortex Search services"], "hash": "f906d7e0cb022046cf76ad6e0cdbaf36"},
    {"q": "What types of data sources can Cortex Search Service be built on?", "options": ["A) Only PDF documents", "B) Only structured tables", "C) Only text files", "D) Both structured tables and unstructured documents"], "hash": "2bb61363abe360dcedf6de42bb13ae01"},
    {"q": "What makes Cortex Search effective for building chatbots over activity reviews?", "options": ["A) It only uses exact keyword matching", "B) Hybrid search combining semantic understanding with keyword matching", "C) It requires manual index rebuilding", "D) It only works with numeric data"], "hash": "0786367b14cb24e2297f71fb04d1b8d9"},
]

results_map = {}
for qi, item in enumerate(quiz_data):
    results_map[qi] = {}
    for opt in item["options"]:
        letter = opt[0]
        escaped_opt = opt.replace("'", "''")
        result = session.sql(f"CALL genai_db.resources.quiz_temp('{item['hash']}', '{escaped_opt}', 'False')").collect()
        feedback = result[0][0]
        is_correct = 'Correct' in feedback or '✅' in feedback
        results_map[qi][letter] = (feedback, is_correct)

html = """<style>
.cq { margin: 20px 0; padding: 16px; border: 1px solid #d0d0d0; border-radius: 10px; background: #fafafa; font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, "Helvetica Neue", Arial, sans-serif; font-size: 14px; }
.cq h4 { font-family: inherit; }
.cq input[type="radio"] { display: none; }
.cq .lbl { display: block; padding: 8px 12px; border-radius: 6px; cursor: pointer; font-family: inherit; }
.cq .lbl:hover { background: #e8f0fe; }
.cq input[type="radio"]:checked + .lbl { border-color: #1a73e8; background: #e8f0fe; font-weight: 600; }
.cq .fb { display: none; padding: 6px 12px; margin-top: 2px; border-radius: 4px; font-weight: 600; font-family: inherit; }
.cq input[type="radio"]:checked + .lbl + .fb { display: block; }
.cq .fb.ok { background: #e6f4ea; color: #1e7e34; }
.cq .fb.no { background: #fce8e6; color: #c62828; }
</style>"""

for qi, item in enumerate(quiz_data):
    html += f'<div class="cq"><h4>Q{qi+1}: {item["q"]}</h4>'
    for opt in item["options"]:
        letter = opt[0]
        feedback, is_correct = results_map[qi][letter]
        css_class = 'ok' if is_correct else 'no'
        uid = f'cq{qi}_{letter}'
        html += f'<div class="opt"><input type="radio" name="cq{qi}" id="{uid}">'
        html += f'<label class="lbl" for="{uid}">{opt}</label>'
        html += f'<div class="fb {css_class}">{letter}) {feedback}</div></div>'
    html += '</div>'

display(HTML(html))

## Key Takeaways

❄️ Cortex Search works with both unstructured documents (Part 1) and structured tables with free-text columns.

❄️ Cortex Search Service is simple to build using SQL and automatically updates the search index via change tracking.

❄️ The Cortex Search Engine is a hybrid search engine that combines lexical search, vector search, and semantic reranking for accurate retrieval.

❄️ Use **SEARCH_PREVIEW** to verify search results, and combine it with **CORTEX.COMPLETE** for a full RAG (Retrieval Augmented Generation) pipeline directly in SQL.

❄️ Use **SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER** to chunk large text columns for better search quality when text exceeds 512 tokens (~2048 characters).

❄️ **CORTEX_SEARCH_BATCH** enables high-throughput offline use cases such as activity matching, recommendations, audience segmentation, and bulk enrichment, by running many queries in parallel against a search service.

❄️ Batch search supports filters (using `@eq`, `@or`) to scope results to specific categories, and uses `METADATA$RANK` to control how many results are returned per query.